# 多模式资源受限项目调度问题 (MRCPSP)

**类别：** 调度

来源：[https://www.hexaly.com/templates/multi-mode-resource-constrained-project-scheduling-mrcpsp](https://www.hexaly.com/templates/multi-mode-resource-constrained-project-scheduling-mrcpsp)


## 问题描述

在多模式资源受限项目调度问题 (MRCPSP) 中，一个项目由一组需要调度的任务组成。对每个任务，其执行过程不可中断，且可以采用不同的执行模式。任务之间存在紧前关系约束：每个任务必须在其所有后继任务开始之前完成。问题中有一组资源，分为可再生资源（任务处理完成后所有资源消耗会归还）和不可再生资源。对每种模式，每种任务执行都有给定的持续时间，并消耗给定数量的资源或权重（可能为零）。每种资源具有给定的最大容量：可以同时处理多个任务，但被处理任务权重之和不得超过该最大容量。

目标是找到一个使 makespan（所有任务处理完成的时间）最小的调度方案。

	

### 建模要点

- 添加 [可选区间决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html#optional-intervals) 来建模任务及其模式
- 使用 [‘hull’ 算子](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#hull) 独立于模式管理任务
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来建模累计资源约束


## 数据

我们为多模式资源受限项目调度问题 (MRCPSP) 提供的实例来自 [PSPLIB instances](https://www.om-db.wi.tum.de/psplib/getdata_mm.html)。数据文件的格式如下：

- 第 6 行包含作业或任务的数量。
- 第 9 行包含可再生资源的数量。
- 第 10 行包含不可再生资源的数量。
- 从第 19 行开始，每个任务占一行，按顺序包含以下整数数据：

- 任务 ID
- 该任务的执行模式数量
- 该任务的后继任务数量
- 该任务的后继任务 ID
- 然后，从下方第 5 行开始，对任务和模式的每个组合各占一行。每行按顺序包含与该配置（任务，模式）相关的整数数据：

- 任务 ID
- 模式 ID
- 该配置下的持续时间
- 该配置下每种资源的消耗权重
- 最后，再下方第 4 行，给出一行包含每种资源的最大容量。


## 程序

多模式资源受限项目调度问题 (MRCPSP) 的 Hexaly 模型是 [RCPSP 模型](https://www.hexaly.com/docs/last/exampletour/resource-constrained-project-scheduling-problem-rcpsp.html) 的扩展。它使用 [可选区间决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html#optional-intervals) 来表示不同执行模式下的任务。我们使用 [presence 算子](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#presence) 来检测每个任务的激活模式。最后，我们使用 ‘hull’ 算子独立于执行模式管理任务。

对于激活模式，每个区间的长度等于该任务在此模式下执行的持续时间。

然后，我们编写紧前关系约束：每个任务必须在其任何后继任务开始之前结束。

我们确保每个任务恰好有一种激活模式。这保证了所有任务都会被调度。

可再生资源约束可以表述如下：对每种可再生资源以及每个时间段，被处理任务所消耗的资源总量不得超过资源容量。为了建模这些约束，我们对每种资源和每个时间段，将所有激活模式下所有激活任务的权重求和。我们使用可变参数 ‘and’ 形式结合 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来确保任何时刻都满足资源容量限制。得益于这种可变参数 ‘and’ 形式，即使时间范围很大，约束的表述仍然紧凑而高效。

不可再生资源约束由涉及每个配置（任务，模式）的 presence 的线性约束建模。

最后，目标是使 makespan 最小化。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def main(instance_file, output_file, time_limit):
    nb_tasks, nb_renewable_resources, nb_resources, nb_modes, capacity, duration, weight, \
        nb_successors, successors, horizon = read_input(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        # Declare the optimization model
        model = optimizer.model

        # Optional interval decisions: time range for each task and mode
        tasks_in_mode = [[model.optional_interval(0, horizon) for m in range(nb_modes[i])] for i in range(nb_tasks)]
        present_mode = [[model.presence(tasks_in_mode[i][m]) for m in range(nb_modes[i])] for i in range(nb_tasks)]

        # Hull
        tasks = [model.hull(tasks_in_mode[task][mode] for mode in range(nb_modes[task])) for task in range(nb_tasks)]

        # Constraints: Task duration
        for task in range(nb_tasks):
            for mode in range(nb_modes[task]):
                model.constraint(model.iif(
                        present_mode[task][mode],
                        model.eq(model.length(tasks_in_mode[task][mode]), duration[task][mode]),
                        1
                ))

        # Constraints: Precedence between tasks
        for task in range(nb_tasks):
            for s in range(nb_successors[task]):
                model.constraint(model.lt(tasks[task], tasks[successors[task][s]]))

        # Constraints: Exactly one active mode for each task
        for task in range(nb_tasks):
            model.constraint(model.eq(sum(present_mode[task]), 1))

        # Makespan: end of the last task
        makespan = model.max([model.end(tasks[task]) for task in range(nb_tasks)])

        # Constraints: Renewable resources
        def capacity_respected(resource, time):
            total_weight = model.sum()
            for task in range(nb_tasks):
                for mode in range(nb_modes[task]):
                    total_weight.add_operand(
                        weight[task][resource][mode]
                        * model.contains(tasks_in_mode[task][mode], time)
                    )
            return model.leq(total_weight, capacity[resource])

        for resource in range(nb_renewable_resources):
            capacity_respected_lambda = model.lambda_function(lambda time: capacity_respected(resource, time))
            model.constraint(model.and_(model.range(makespan), capacity_respected_lambda))

        # Constraints: Non-renewable resources
        for resource in range(nb_renewable_resources, nb_resources):
            total_weight = model.sum()
            for task in range(nb_tasks):
                for mode in range(nb_modes[task]):
                    total_weight.add_operand(weight[task][resource][mode] * present_mode[task][mode])
            
            model.constraint(model.leq(total_weight, capacity[resource]))

        # Objective: Minimize the makespan
        model.minimize(makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - total makespan
        # - for each task, the task ID, the mode ID, the start and end times
        #
        if output_file != None:
            with open(output_file, "w") as file:
                print("Solution written in file", output_file)
                file.write(str(makespan.value) + "\n")
                for task in range(nb_tasks):
                    activeModeId = -1
                    for mode in range(nb_modes[task]):
                        if (present_mode[task][mode].value):
                            activeModeId = mode
                            break
                    file.write(str(task + 1) + " " + str(activeModeId + 1) + " "
                            + str(tasks[task].value.start()) + " "
                            + str(tasks[task].value.end()))
                    file.write("\n")

def read_input(filename):
    with open(filename) as file:
        lines = file.readlines()

    ## Parse number of tasks
    line = lines[5].split(":")
    nb_tasks = int(line[1])

    ## Parse number of resources
    line = lines[8].split(":")
    nb_renewable_resources = int(line[1].split()[0])

    line = lines[9].split(":")
    nb_non_renewable_resources = int(line[1].split()[0])
    nb_resources = nb_renewable_resources + nb_non_renewable_resources

    # Number of available modes for each task
    nb_modes = [0 for i in range(nb_tasks)]
    
    # Number of successors of each task
    nb_successors = [0 for i in range(nb_tasks)]

    # Successors of each task
    successors = [[] for i in range(nb_tasks)]

    ## Parse successors of each task
    line_index = 18
    line = lines[line_index].split()
    task_id = int(line[0]) - 1

    while True:
        nb_modes[task_id] = int(line[1])
        nb_successors[task_id] = int(line[2])
        successors[task_id] = [int(line[3 + s]) - 1 for s in range(nb_successors[task_id])]
        if task_id + 1 == nb_tasks:
            break
        line_index = line_index + 1
        line = lines[line_index].split()
        task_id = int(line[0]) - 1
    
    ## Parse tasks durations per mode AND consumed resource weight per mode for each task
    line_index = line_index + 5
    
    # Duration of each task and mode
    duration = [[] for i in range(nb_tasks)]

    # Required resource weight for each task and mode
    weight = [[[]] for i in range(nb_tasks)]

    for task in range(nb_tasks):
        weight[task] = [[0 for m in range(nb_modes[task])] for r in range(nb_resources)]
        duration[task] = [0 for m in range(nb_modes[task])]
        for mode in range(nb_modes[task]):
            line = lines[line_index].split()
            base_index = 1 if mode == 0 else 0
            duration[task][mode] = int(line[base_index + 1])
            for resource in range(nb_resources):
                weight[task][resource][mode] = int(line[base_index + 2 + resource])
            line_index = line_index + 1

    # Maximum capacity of each resource
    capacity = [int(lines[line_index + 3].split()[r]) for r in range(nb_resources)]

    # Trivial upper bound for the start times of the tasks
    horizon = sum(max(duration[task][mode] for mode in range(nb_modes[task])) for task in range(nb_tasks))

    return (nb_tasks, nb_renewable_resources, nb_resources, nb_modes, capacity, duration, weight, nb_successors, successors, horizon)

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python rcpsp_multi_mode.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
